<a href="https://colab.research.google.com/github/MiguelFigueroa02/ITAM_Sport_Lucho_FC/blob/main/notebooks/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import re

## 1. Carga de datos
---
A partir de los archivos json disponibles, procesamos los datos para construir un dataset que contenga todos los partidos del américa disponibles en el periodo 2023 - 2026

In [4]:
DATA_DIR = Path("/content/")
OUTPUT_FILE = Path("/content/america_matches.csv")
AMERICA_ID = 1229

In [5]:
json_files = sorted(DATA_DIR.glob("matches-*.json"))

In [6]:
if not json_files:
    raise FileNotFoundError(
        f"No se encontraron archivos JSON en: {DATA_DIR.resolve()}"
    )

In [7]:
records = []

for file in json_files:

    print(f"\nArchivo: {file.name}")

    with open(file, "r", encoding="utf-8") as f:
        matches = json.load(f)

    print(f"Partidos encontrados en archivo: {len(matches)}")

    for match in matches:

        home = match["home_team"]
        away = match["away_team"]

        home_id = home["home_team_id"]
        away_id = away["away_team_id"]

        if home_id != AMERICA_ID and away_id != AMERICA_ID:
            continue

        is_home = home_id == AMERICA_ID

        opponent = (
            away["away_team_name"]
            if is_home
            else home["home_team_name"]
        )

        america_score = (
            match["home_score"]
            if is_home
            else match["away_score"]
        )

        opponent_score = (
            match["away_score"]
            if is_home
            else match["home_score"]
        )


        managers = home.get("managers", []) if is_home else away.get("managers", [])

        coach = (
            managers[0]["name"]
            if managers
            else None
        )


        records.append({

            "match_id": match["match_id"],

            "match_date": match["match_date"],

            "kick_off": match.get("kick_off"),

            "season": match["season"]["season_name"],

            "competition": match["competition"]["competition_name"],

            "competition_stage": (
                match.get("competition_stage", {}).get("name")
            ),

            "match_week": match.get("match_week"),

            "opponent": opponent,

            "venue": "Home" if is_home else "Away",

            "america_score": america_score,

            "opponent_score": opponent_score,

            "coach": coach,

            "stadium": (
                match.get("stadium", {}).get("name")
            ),

            "match_status": match.get("match_status"),

            "match_status_360": match.get("match_status_360"),

        })



america_matches = pd.DataFrame(records)


if america_matches.empty:
    raise ValueError(
        "No se encontraron partidos del América. "
        "Revisa el AMERICA_ID."
    )


def get_result(row):

    if row["america_score"] > row["opponent_score"]:
        return "W"

    elif row["america_score"] == row["opponent_score"]:
        return "D"

    else:
        return "L"


america_matches["result"] = america_matches.apply(
    get_result,
    axis=1
)


america_matches["match_date"] = pd.to_datetime(
    america_matches["match_date"]
)


america_matches = america_matches.sort_values(
    ["match_date", "match_id"]
).reset_index(drop=True)



OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)



america_matches.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 60)
print("CATÁLOGO DEL AMÉRICA")
print("=" * 60)

print(f"\nTotal de partidos: {len(america_matches)}")

print("\nPartidos por temporada:")
print(
    america_matches
    .groupby("season")
    .size()
    .sort_index()
)

print("\nResultados:")
print(
    america_matches["result"]
    .value_counts()
)

print("\nLocal / Visitante:")
print(
    america_matches["venue"]
    .value_counts()
)

print("\nDisponibilidad de 360:")
print(
    america_matches["match_status_360"]
    .value_counts(dropna=False)
)

print("\nEntrenadores:")
print(
    america_matches["coach"]
    .value_counts(dropna=False)
)

print(f"\nArchivo generado:")
print(OUTPUT_FILE.resolve())

print("\nPrimeros registros:")
print(
    america_matches.head(10).to_string(index=False)
)


Archivo: matches-2023-2024.json
Partidos encontrados en archivo: 340

Archivo: matches-2024-2025.json
Partidos encontrados en archivo: 340

Archivo: matches-2025-2026.json
Partidos encontrados en archivo: 337

CATÁLOGO DEL AMÉRICA

Total de partidos: 131

Partidos por temporada:
season
2023/2024    46
2024/2025    47
2025/2026    38
dtype: int64

Resultados:
result
W    69
D    37
L    25
Name: count, dtype: int64

Local / Visitante:
venue
Away    66
Home    65
Name: count, dtype: int64

Disponibilidad de 360:
match_status_360
available      130
unscheduled      1
Name: count, dtype: int64

Entrenadores:
coach
André Soares Jardine              129
Diego Alberto Cervantes Chávez      2
Name: count, dtype: int64

Archivo generado:
/content/america_matches.csv

Primeros registros:
 match_id match_date     kick_off    season competition competition_stage  match_week    opponent venue  america_score  opponent_score                coach                stadium match_status match_status_360 r

## 2.  Auditoría inicial de los datos de los eventos de partidos.
---

Hacemos en primera instancia una evaluación inicial de los datos.

## 3. Integración y exploración de los datos

In [8]:
def evaluacion(archivo):
  repo = 'https://raw.githubusercontent.com/MiguelFigueroa02/ITAM_Sport_Lucho_FC/refs/heads/main/data/raw/'
  path = repo + archivo
  CHUNK_SIZE = 50_000
  print("### Evaluación de eventos")
  header = pd.read_csv(
      path,
      nrows=0
    )
  columns = header.columns.tolist()
  print(f"\nNúmero de columnas: {len(columns)}")
  print("\nColumnas:")
  for i, col in enumerate(columns, start=1):
      print(f"{i:3}. {col}")

  total_rows = 0
  unique_matches = set()

  sample = None

  for chunk in pd.read_csv(
      path,
      chunksize=CHUNK_SIZE,
      low_memory=False,
      on_bad_lines='skip'
  ):

      total_rows += len(chunk)

      if sample is None:
          sample = chunk.head(5).copy()

      if "match_id" in chunk.columns:
          unique_matches.update(
              chunk["match_id"].dropna().unique()
          )

  print("### RESUMEN")

  print(f"\nFilas totales: {total_rows:,}")

  print(f"Partidos únicos: {len(unique_matches):,}")

  print("### MUESTRA DE EVENTOS")

  print(
    sample.to_csv(index=False)
  )
  print("### TIPOS DE DATOS")

  print(sample.dtypes)


In [9]:
evaluacion('Liga_MX_2023_2024_America_p1.csv')

### Evaluación de eventos

Número de columnas: 251

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [10]:
evaluacion('Liga_MX_2023_2024_America_p2.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [11]:
evaluacion('Liga_MX_2023_2024_America_p3.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [12]:
evaluacion('Liga_MX_2024_2025_America_p1.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [13]:
evaluacion('Liga_MX_2024_2025_America_p2.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [14]:
evaluacion('Liga_MX_2024_2025_America_p3.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [15]:
evaluacion('Liga_MX_2025_2026_America_p1.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

In [16]:
evaluacion('Liga_MX_2025_2026_America_p2.csv')

### Evaluación de eventos

Número de columnas: 254

Columnas:
  1. id
  2. index
  3. match_id
  4. period
  5. timestamp
  6. minute
  7. second
  8. event_type_id
  9. event_type_name
 10. possession
 11. possession_team_id
 12. possession_team_name
 13. play_pattern_id
 14. play_pattern_name
 15. team_id
 16. team_name
 17. player_id
 18. player_name
 19. player_position_id
 20. player_position_name
 21. location_x
 22. location_y
 23. end_location_x
 24. end_location_y
 25. end_location_z
 26. impact_height
 27. duration
 28. under_pressure
 29. counterpress
 30. substituted_player_id
 31. substituted_player_name
 32. pass_recipient_id
 33. pass_recipient_name
 34. pass_length
 35. pass_angle
 36. pass_height_id
 37. pass_height_name
 38. pass_cross
 39. pass_cut_back
 40. pass_switch
 41. assisted_shot_id
 42. key_pass_id
 43. follows_dribble
 44. one_on_one
 45. open_goal
 46. statsbomb_xg
 47. off_camera
 48. out
 49. obv_for_after
 50. obv_for_before
 51. obv_for_net
 52. obv_a

### Construcción de archivo RAW

In [17]:
repo = 'https://raw.githubusercontent.com/MiguelFigueroa02/ITAM_Sport_Lucho_FC/refs/heads/main/data/raw/'

In [18]:
datasets = [
    "Liga_MX_2023_2024_America_p1.csv",
    "Liga_MX_2023_2024_America_p2.csv",
    "Liga_MX_2023_2024_America_p3.csv",
    "Liga_MX_2024_2025_America_p1.csv",
    "Liga_MX_2024_2025_America_p2.csv",
    "Liga_MX_2024_2025_America_p3.csv",
    "Liga_MX_2025_2026_America_p1.csv",
    "Liga_MX_2025_2026_America_p2.csv"
]

In [19]:
for dataset in datasets:
    file = repo + dataset
    df = pd.read_csv(file, nrows=5)

    print(f"{dataset}")
    print(f"  Filas inspeccionadas: {len(df)}")
    print(f"  Columnas: {len(df.columns)}")
    # print(f"  Tamaño: {file.stat().st_size / 1024**2:.2f} MB")
    print()

Liga_MX_2023_2024_America_p1.csv
  Filas inspeccionadas: 5
  Columnas: 251

Liga_MX_2023_2024_America_p2.csv
  Filas inspeccionadas: 5
  Columnas: 254

Liga_MX_2023_2024_America_p3.csv
  Filas inspeccionadas: 5
  Columnas: 254

Liga_MX_2024_2025_America_p1.csv
  Filas inspeccionadas: 5
  Columnas: 254

Liga_MX_2024_2025_America_p2.csv
  Filas inspeccionadas: 5
  Columnas: 254

Liga_MX_2024_2025_America_p3.csv
  Filas inspeccionadas: 5
  Columnas: 254

Liga_MX_2025_2026_America_p1.csv
  Filas inspeccionadas: 5
  Columnas: 254

Liga_MX_2025_2026_America_p2.csv
  Filas inspeccionadas: 5
  Columnas: 254



In [20]:
schemas = {}

for dataset in datasets:
    file = repo + dataset
    df = pd.read_csv(file, nrows=0)
    schemas[dataset] = set(df.columns)

all_columns = set().union(*schemas.values())

print(f"Columnas totales distintas: {len(all_columns)}\n")

for filename, columns in schemas.items():
    missing = all_columns - columns
    extra = columns - min(schemas.values(), key=len)

    print(f"=== {filename} ===")
    print(f"Columnas: {len(columns)}")
    print(f"Faltantes respecto al universo: {len(missing)}")

    if missing:
        print("Faltantes:", sorted(missing))

    print()

Columnas totales distintas: 707

=== Liga_MX_2023_2024_America_p1.csv ===
Columnas: 251
Faltantes respecto al universo: 456
Faltantes: ['107179_defensive_responsibility_probability', '107179_interception_like_defensive_responsibility_probability', '107179_pressure_like_defensive_responsibility_probability', '10846_defensive_responsibility_probability', '10846_interception_like_defensive_responsibility_probability', '10846_pressure_like_defensive_responsibility_probability', '123690_defensive_responsibility_probability', '123690_interception_like_defensive_responsibility_probability', '123690_pressure_like_defensive_responsibility_probability', '130953_defensive_responsibility_probability', '130953_interception_like_defensive_responsibility_probability', '130953_pressure_like_defensive_responsibility_probability', '131123_defensive_responsibility_probability', '131123_interception_like_defensive_responsibility_probability', '131123_pressure_like_defensive_responsibility_probability', '1

In [21]:
# Tomamos como referencia uno de los archivos de 254 columnas
reference = datasets[0]
reference_file = repo + reference

reference_columns = set(
    pd.read_csv(reference_file, nrows=0).columns
)

for dataset in datasets[1:]:
    file = repo + dataset
    columns = set(pd.read_csv(file, nrows=0).columns)

    missing = reference_columns - columns
    extra = columns - reference_columns

    print(f'## {dataset}')
    print(f"Columnas: {len(columns)}")

    print(f"\nFaltantes respecto a {reference}: {len(missing)}")
    if missing:
        print(sorted(missing))

    print(f"\nExtra respecto a {reference}: {len(extra)}")
    if extra:
        print(sorted(extra))

## Liga_MX_2023_2024_America_p2.csv
Columnas: 254

Faltantes respecto a Liga_MX_2023_2024_America_p1.csv: 90
['11388_defensive_responsibility_probability', '11388_interception_like_defensive_responsibility_probability', '11388_pressure_like_defensive_responsibility_probability', '13034_defensive_responsibility_probability', '13034_interception_like_defensive_responsibility_probability', '13034_pressure_like_defensive_responsibility_probability', '131162_defensive_responsibility_probability', '131162_interception_like_defensive_responsibility_probability', '131162_pressure_like_defensive_responsibility_probability', '134157_defensive_responsibility_probability', '134157_interception_like_defensive_responsibility_probability', '134157_pressure_like_defensive_responsibility_probability', '182089_defensive_responsibility_probability', '182089_interception_like_defensive_responsibility_probability', '182089_pressure_like_defensive_responsibility_probability', '23842_defensive_responsibility

#### Interpretación
---
- Estamos encontrando que cada uno de los datasets presenta columnas dinámicas asociadas al ID de los jugadores de cada partido.
- Para continuar con nuestro proceso, procederemos a definir cuáles columnas son estables y cuáles son asociados a resultados dinámicos. A partir de las columnas estables son de las que en primera instancia procederemos a establecer métricas básicas asociadas al rendimiento para interpretación del estilo táctico.


In [22]:
columns = pd.read_csv(reference_file, nrows=0).columns

dynamic_columns = [
    col for col in columns
    if re.match(r"^\d+_", col)
]

stable_columns = [
    col for col in columns
    if col not in dynamic_columns
]

print(f"Columnas totales: {len(columns)}")
print(f"Columnas dinámicas de jugador: {len(dynamic_columns)}")
print(f"Columnas estables: {len(stable_columns)}")

print("\nPrimeras columnas estables:")
print(stable_columns[:50])

Columnas totales: 251
Columnas dinámicas de jugador: 123
Columnas estables: 128

Primeras columnas estables:
['id', 'index', 'match_id', 'period', 'timestamp', 'minute', 'second', 'event_type_id', 'event_type_name', 'possession', 'possession_team_id', 'possession_team_name', 'play_pattern_id', 'play_pattern_name', 'team_id', 'team_name', 'player_id', 'player_name', 'player_position_id', 'player_position_name', 'location_x', 'location_y', 'end_location_x', 'end_location_y', 'end_location_z', 'impact_height', 'duration', 'under_pressure', 'counterpress', 'substituted_player_id', 'substituted_player_name', 'pass_recipient_id', 'pass_recipient_name', 'pass_length', 'pass_angle', 'pass_height_id', 'pass_height_name', 'pass_cross', 'pass_cut_back', 'pass_switch', 'assisted_shot_id', 'key_pass_id', 'follows_dribble', 'one_on_one', 'open_goal', 'statsbomb_xg', 'off_camera', 'out', 'obv_for_after', 'obv_for_before']


In [23]:
# Comparar las columnas estables entre todos los archivos

reference_columns = set(stable_columns)

for dataset in datasets:
    file = repo + dataset
    columns = set(pd.read_csv(file, nrows=0).columns)

    dynamic = {
        col for col in columns
        if re.match(r"^\d+_", col)
    }

    stable = columns - dynamic

    missing = reference_columns - stable
    extra = stable - reference_columns

    print(f"\n{'='*70}")
    print(dataset)
    print(f"Columnas estables: {len(stable)}")
    print(f"Faltantes: {len(missing)}")
    print(f"Extras: {len(extra)}")

    if missing:
        print("Faltantes:", sorted(missing))

    if extra:
        print("Extras:", sorted(extra))


Liga_MX_2023_2024_America_p1.csv
Columnas estables: 128
Faltantes: 0
Extras: 0

Liga_MX_2023_2024_America_p2.csv
Columnas estables: 128
Faltantes: 0
Extras: 0

Liga_MX_2023_2024_America_p3.csv
Columnas estables: 128
Faltantes: 0
Extras: 0

Liga_MX_2024_2025_America_p1.csv
Columnas estables: 128
Faltantes: 0
Extras: 0

Liga_MX_2024_2025_America_p2.csv
Columnas estables: 128
Faltantes: 0
Extras: 0

Liga_MX_2024_2025_America_p3.csv
Columnas estables: 128
Faltantes: 0
Extras: 0

Liga_MX_2025_2026_America_p1.csv
Columnas estables: 128
Faltantes: 0
Extras: 0

Liga_MX_2025_2026_America_p2.csv
Columnas estables: 128
Faltantes: 0
Extras: 0


#### Concatenación
---
A partir de las columnas estables detectadas, procederemos a hacer la configuración de un dataset maestro


In [24]:
stable_columns = list(stable_columns)

dfs = []

for dataset in datasets:
    file = repo + dataset
    print(f"Cargando: {dataset}")

    df = pd.read_csv(file, usecols=stable_columns)

    # Trazabilidad
    df["source_file"] = dataset

    dfs.append(df)


# Concatenar los 8 archivos
events_raw = pd.concat(
    dfs,
    ignore_index=True
)

print("## DATASET CONSOLIDADO")

print(f"Filas: {len(events_raw):,}")
print(f"Columnas: {len(events_raw.columns)}")
print(f"Partidos únicos: {events_raw['match_id'].nunique()}")

print("\nPartidos por archivo:")
print(
    events_raw.groupby("source_file")["match_id"]
    .nunique()
)

print("\nMemoria utilizada:")
print(f"{events_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Cargando: Liga_MX_2023_2024_America_p1.csv


/tmp/ipykernel_28079/898783433.py:9: DtypeWarning: Columns (60,97,99,246,247,249) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, usecols=stable_columns)


Cargando: Liga_MX_2023_2024_America_p2.csv


/tmp/ipykernel_28079/898783433.py:9: DtypeWarning: Columns (60,90,92,93,97,99) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, usecols=stable_columns)


Cargando: Liga_MX_2023_2024_America_p3.csv


...

Cargando: Liga_MX_2024_2025_America_p2.csv
Cargando: Liga_MX_2024_2025_America_p3.csv


/tmp/ipykernel_28079/898783433.py:9: DtypeWarning: Columns (17,19,30,32,36,37,38,39,40,41,43,44,56,58,60,62,64,66,88,90,92,93,101,102,104,249,250,253) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, usecols=stable_columns)


Cargando: Liga_MX_2025_2026_America_p1.csv


/tmp/ipykernel_28079/898783433.py:9: DtypeWarning: Columns (249,253) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, usecols=stable_columns)


Cargando: Liga_MX_2025_2026_America_p2.csv
## DATASET CONSOLIDADO
Filas: 501,944
Columnas: 129
Partidos únicos: 131

Partidos por archivo:
source_file
Liga_MX_2023_2024_America_p1.csv    20
Liga_MX_2023_2024_America_p2.csv    20
Liga_MX_2023_2024_America_p3.csv     6
Liga_MX_2024_2025_America_p1.csv    20
Liga_MX_2024_2025_America_p2.csv    20
Liga_MX_2024_2025_America_p3.csv     7
Liga_MX_2025_2026_America_p1.csv    20
Liga_MX_2025_2026_America_p2.csv    18
Name: match_id, dtype: int64

Memoria utilizada:
1119.55 MB


Comparación de datos generados a partir del dataset procesado de partidos

In [25]:
partidos = 'https://raw.githubusercontent.com/MiguelFigueroa02/ITAM_Sport_Lucho_FC/refs/heads/main/data/processed/america_matches.csv'
matches = pd.read_csv(partidos)

print("Dimensiones:", matches.shape)

print("\nColumnas:")
print(matches.columns.tolist())

print("\nPrimeras filas:")
display(matches.head().to_markdown())

print("\nPartidos únicos:")
print(matches["match_id"].nunique())

print("\nDuplicados de match_id:")
print(matches["match_id"].duplicated().sum())

print("\nTipos de datos:")
print(matches.dtypes)

Dimensiones: (131, 16)

Columnas:
['match_id', 'match_date', 'kick_off', 'season', 'competition', 'competition_stage', 'match_week', 'opponent', 'venue', 'america_score', 'opponent_score', 'coach', 'stadium', 'match_status', 'match_status_360', 'result']

Primeras filas:


'|    |   match_id | match_date   | kick_off     | season    | competition   | competition_stage   |   match_week | opponent   | venue   |   america_score |   opponent_score | coach                | stadium         | match_status   | match_status_360   | result   |\n|---:|-----------:|:-------------|:-------------|:----------|:--------------|:--------------------|-------------:|:-----------|:--------|----------------:|-----------------:|:---------------------|:----------------|:---------------|:-------------------|:---------|\n|  0 |    3889637 | 2023-07-01   | 01:00:00.000 | 2023/2024 | Liga MX       | Apertura            |            1 | Juárez     | Home    |               1 |                2 | André Soares Jardine | Estadio Azteca  | available      | available          | L        |\n|  1 |    3889660 | 2023-07-16   | 01:00:00.000 | 2023/2024 | Liga MX       | Apertura            |            3 | Puebla     | Home    |               3 |                0 | André Soares Jardine | Est


Partidos únicos:
131

Duplicados de match_id:
0

Tipos de datos:
match_id              int64
match_date           object
kick_off             object
season               object
competition          object
competition_stage    object
match_week            int64
opponent             object
venue                object
america_score         int64
opponent_score        int64
coach                object
stadium              object
match_status         object
match_status_360     object
result               object
dtype: object


In [26]:
event_match_ids = set(events_raw["match_id"].unique())
match_match_ids = set(matches["match_id"].unique())

print("Partidos en eventos:", len(event_match_ids))
print("Partidos en catálogo:", len(match_match_ids))

print("\nEventos sin metadata:")
print(event_match_ids - match_match_ids)

print("\nPartidos del catálogo sin eventos:")
print(match_match_ids - event_match_ids)

Partidos en eventos: 131
Partidos en catálogo: 131

Eventos sin metadata:
set()

Partidos del catálogo sin eventos:
set()


Una vez que comprobamos que los partidos de ambos datasets coinciden, procederemos a hacer un join para generar nuestro dataset maestro

In [27]:
events = events_raw.merge(
    matches,
    on="match_id",
    how="left",
    validate="many_to_one"
)

print("Dimensiones:", events.shape)

print("\nPartidos únicos:", events["match_id"].nunique())

print("\nColumnas de contexto agregadas:")
print([
    "season",
    "match_date",
    "opponent",
    "venue",
    "america_score",
    "opponent_score",
    "coach",
    "result"
])

print("\nValores nulos en contexto:")
print(
    events[
        [
            "season",
            "match_date",
            "opponent",
            "venue",
            "america_score",
            "opponent_score",
            "coach",
            "result"
        ]
    ].isna().sum()
)

Dimensiones: (501944, 144)

Partidos únicos: 131

Columnas de contexto agregadas:
['season', 'match_date', 'opponent', 'venue', 'america_score', 'opponent_score', 'coach', 'result']

Valores nulos en contexto:
season            0
match_date        0
opponent          0
venue             0
america_score     0
opponent_score    0
coach             0
result            0
dtype: int64


Queremos conocer el tipo de eventos asociados a nuestro dataset generado

In [28]:
print("EVENT TYPES")
print(events["event_type_name"].value_counts())

print("\n" + "="*70)
print("PLAY PATTERNS")
print(events["play_pattern_name"].value_counts())

print("\n" + "="*70)
print("POSSESSION TEAM")
print(events["possession_team_name"].value_counts().head(10))

print("\n" + "="*70)
print("RESULTADOS")
print(events["result"].value_counts())

print("\n" + "="*70)
print("LOCALÍA")
print(events["venue"].value_counts())

EVENT TYPES
event_type_name
Pass                 126618
Ball Receipt*        116392
Carries              103365
Shot                  47888
Pressure              41287
Ball Recovery         11619
Duel                   7793
Tactical Shift         5734
Clearance              4611
Block                  4542
Goal Keeper            3974
Miscontrol             3475
Dribble                3321
Foul Committed         3269
Starting XI            3144
Foul Won               3084
Dispossessed           3078
Dribbled Past          1901
Interception           1878
Substitution           1177
Injury Stoppage         824
50/50                   665
Half Start              530
Half End                530
Referee Ball-Drop       455
Player Off              186
Player On               185
Shield                  170
Bad Behaviour           134
Error                    62
Offside                  43
Own Goal For              5
Own Goal Against          5
Name: count, dtype: int64

PLAY PATTERNS
play_pa

## 4. Construcción

### Pases por posesión


Vamos a evaluar la cantidad de pases que suelen tener por posesión de manera general. El objetivo de su análisis es caracterizar como primera métrica para la construcción ofensiva del equipo.

In [29]:
TEAM_ID = 1229

In [30]:
def possesions(team_id):
  team_events = events[
      events["possession_team_id"] == team_id
  ].copy()
  team_name = team_events.possession_team_name.unique()[0]

  print(f"Eventos de {team_name}:", len(team_events))
  print("Posesiones:", team_events["possession"].nunique())

  print("\nPrimeras posesiones:")
  print(
      team_events[
          ["match_id", "possession", "event_type_name",
          "player_name", "minute"]
      ].head(20).to_markdown()
  )

  print("\nEvento por posesiones:")
  possession_size = (
      team_events.groupby(["match_id", "possession"]).size().reset_index(name="events_in_possession")
  )

  print(possession_size.head().to_markdown())

  print("\nEstadísticas:")
  print(
      possession_size["events_in_possession"].describe().to_markdown()
  )

  passes = team_events[
      team_events["event_type_name"] == "Pass"
  ].copy()

  print('\n## Pases por posesión\n')
  print(f"Pases de {team_name}:", len(passes))

  passes_per_possession = (
      passes
      .groupby(["match_id", "possession"])
      .size()
      .reset_index(name="passes")
  )

  print("\nPrimeras posesiones:")
  print(passes_per_possession.head(10).to_markdown())

  print("\nEstadísticas de pases por posesión:")
  print(passes_per_possession["passes"].describe().to_markdown())

  print('## VALIDACIÓN: POSESIONES CON Y SIN PASES')

  all_possessions = (
      team_events[["match_id", "possession"]]
      .drop_duplicates()
  )

  possession_check = all_possessions.merge(
     passes_per_possession,
     on=["match_id", "possession"],
     how="left"
  )

  possession_check["passes"] = possession_check["passes"].fillna(0)

  print(
      "Total posesiones:",
      len(possession_check)
  )

  print(
      "Posesiones sin pases:",
      (possession_check["passes"] == 0).sum()
  )

  print(
     "Posesiones con al menos un pase:",
     (possession_check["passes"] > 0).sum()
  )

  print("\nDistribución:")
  print(
     possession_check["passes"].describe().to_markdown()
  )

  print('# Pases por posesión')

  passes_match = (
      possession_check
      .groupby("match_id")
      .agg(
          passes_per_possession=("passes", "mean"),
          possessions=("possession", "count")
      )
      .reset_index()
  )

  # Incorporamos contexto del partido
  passes_match = passes_match.merge(
      matches[
          [
              "match_id",
              "opponent",
              "venue",
              "america_score",
              "opponent_score",
              "result",
              "season",
              "coach"
          ]
      ],
      on="match_id",
      how="left",
      validate="one_to_one"
  )

  print(passes_match.head())

  print("\nDimensiones:")
  print(passes_match.shape)

  print("\nEstadísticas:")
  print(
      passes_match["passes_per_possession"].describe().to_markdown()
  )

  print('\n## Pases por posesión ligados al resultado\n')

  result_summary = (
      passes_match
      .groupby("result")
      ["passes_per_possession"]
      .agg(["count", "mean", "median", "std"])
      .round(2)
  )

  print(result_summary.to_markdown())
  print('\n Pases por posesión relacionados con la localía del equipo\n')

  venue_summary = (
      passes_match
      .groupby("venue")
      ["passes_per_possession"]
      .agg(["count", "mean", "median", "std"])
      .round(2)
  )

  print(venue_summary.to_markdown())

  print('\nAnálisis de temporal de los pases por posesión')

  season_summary = (
      passes_match
      .groupby("season")
      ["passes_per_possession"]
      .agg(["count", "mean", "median", "std"])
      .round(2)
  )

  print(season_summary.to_markdown())

In [31]:
possesions(TEAM_ID)

Eventos de América: 272145
Posesiones: 249

Primeras posesiones:
|    |   match_id |   possession | event_type_name   | player_name                     |   minute |
|---:|-----------:|-------------:|:------------------|:--------------------------------|---------:|
|  4 |    3889637 |            2 | Pass              | Román Martínez                  |        0 |
|  5 |    3889637 |            2 | Ball Receipt*     | Jonathan dos Santos Ramírez     |        0 |
|  6 |    3889637 |            2 | Pass              | Jonathan dos Santos Ramírez     |        0 |
|  7 |    3889637 |            2 | Ball Receipt*     | Richard Rafael Sánchez Guerrero |        0 |
|  8 |    3889637 |            2 | Pass              | Richard Rafael Sánchez Guerrero |        0 |
|  9 |    3889637 |            2 | Ball Receipt*     | Kevin Nahin Álvarez Campos      |        0 |
| 10 |    3889637 |            2 | Carries           | Kevin Nahin Álvarez Campos      |        0 |
| 11 |    3889637 |            2 | 

#### Interpretación
---
- Al comparar la progresión de pases bajos distintos escenarios, no encontramos alguna diferencia significativa que nos indique que los pases en cada posesión sean determinantes para el resultado.
- Dado los resultados obtenidos, la métrica no muestra evidencia para sustentar que sea un parámetro táctico a tomar en cuenta al estilo de juego del América

### Progresión mediante pases
---
Estamos intentando analizar la siguiente pregunta:
- ¿Este equipo consigue avanzar el balón y cambiar de progresión según el contexto?


In [32]:
team_events = events[
      events["possession_team_id"] == TEAM_ID
  ].copy()

In [33]:
passes = team_events[team_events["event_type_name"] == "Pass"].copy()

In [34]:
passes_per_possession = (
    passes.groupby(["match_id", "possession"])
    .size()
    .reset_index(name="passes")
)

In [35]:
all_possessions = (
    team_events[["match_id", "possession"]]
    .drop_duplicates()
)

In [36]:
possession_check = all_possessions.merge(
    passes_per_possession,
    on=["match_id", "possession"],
    how="left"
)

In [37]:
passes_match = (
    possession_check
    .groupby("match_id")
    .agg(
        passes_per_possession=("passes", "mean"),
        possessions=("possession", "count")
    )
    .reset_index()
)

In [38]:
match_id_test = passes_match.iloc[0]["match_id"]

In [39]:
match_test = team_events[
    team_events["match_id"] == match_id_test
].copy()

In [40]:
test = match_test[[
    "period",
    "minute",
    "event_type_name",
    "player_name",
    "location_x",
    "location_y",
    "end_location_x",
    "end_location_y"
]]
print(test.head(30).to_markdown(index=False))

|   period |   minute | event_type_name   | player_name                     |   location_x |   location_y |   end_location_x |   end_location_y |
|---------:|---------:|:------------------|:--------------------------------|-------------:|-------------:|-----------------:|-----------------:|
|        1 |        0 | Pass              | Román Martínez                  |         61   |         40.1 |             47.8 |             36.8 |
|        1 |        0 | Ball Receipt*     | Jonathan dos Santos Ramírez     |         47.8 |         36.8 |            nan   |            nan   |
|        1 |        0 | Pass              | Jonathan dos Santos Ramírez     |         47.6 |         37.4 |             46.5 |             49.5 |
|        1 |        0 | Ball Receipt*     | Richard Rafael Sánchez Guerrero |         46.5 |         49.5 |            nan   |            nan   |
|        1 |        0 | Pass              | Richard Rafael Sánchez Guerrero |         45.9 |         49.5 |             39.7

In [41]:
passes_test = match_test[
    match_test["event_type_name"] == "Pass"
].copy()

print("\nPases por periodo:")
print(
    passes_test
    .groupby("period")
    [["location_x", "end_location_x"]]
    .agg(["min", "mean", "max"]).to_markdown()
)


Pases por periodo:
|   period |   ('location_x', 'min') |   ('location_x', 'mean') |   ('location_x', 'max') |   ('end_location_x', 'min') |   ('end_location_x', 'mean') |   ('end_location_x', 'max') |
|---------:|------------------------:|-------------------------:|------------------------:|----------------------------:|-----------------------------:|----------------------------:|
|        1 |                     6   |                  55.9326 |                     120 |                         6.2 |                      59.3264 |                         120 |
|        2 |                     4.6 |                  57.4817 |                     120 |                         5   |                      61.5598 |                         120 |


In [42]:
[
    col for col in events.columns
    if any(
        term in col.lower()
        for term in ["angle", "direction", "orientation", "rotation"]
    )
]

['pass_angle', 'attacking_direction']

In [43]:
[
    col for col in events.columns
    if col.startswith("pass_")
]

['pass_recipient_id',
 'pass_recipient_name',
 'pass_length',
 'pass_angle',
 'pass_height_id',
 'pass_height_name',
 'pass_cross',
 'pass_cut_back',
 'pass_switch',
 'pass_cluster_id',
 'pass_cluster_label',
 'pass_cluster_probability',
 'pass_success_probability']

Analizamos la variable attacking_directions

In [44]:
print(
    team_events["attacking_direction"]
    .value_counts(dropna=False).to_markdown()
)

| attacking_direction   |   count |
|:----------------------|--------:|
| right_to_left         |  136948 |
| left_to_right         |  134829 |
| nan                   |     368 |


In [45]:
print(
    team_events
    .groupby(["match_id", "period"])["attacking_direction"]
    .first()
    .value_counts(dropna=False).to_markdown()
)

| attacking_direction   |   count |
|:----------------------|--------:|
| left_to_right         |     133 |
| right_to_left         |     132 |


Evaluaremos entonces cuál es la progresión de los pases

In [46]:
passes_progression = passes.copy()

In [47]:
passes_progression = passes_progression.dropna(
    subset=[
        "location_x",
        "end_location_x",
        "attacking_direction"
    ]
)

In [48]:
passes_progression["progression_x"] = np.where(
    passes_progression["attacking_direction"] == "left_to_right",
    passes_progression["end_location_x"] -
        passes_progression["location_x"],
    passes_progression["location_x"] -
        passes_progression["end_location_x"]
)

In [49]:
print(
    passes_progression[
        [
            "match_id",
            "player_name",
            "attacking_direction",
            "location_x",
            "end_location_x",
            "progression_x"
        ]
    ].head(20).to_markdown()
)

|    |   match_id | player_name                     | attacking_direction   |   location_x |   end_location_x |   progression_x |
|---:|-----------:|:--------------------------------|:----------------------|-------------:|-----------------:|----------------:|
|  4 |    3889637 | Román Martínez                  | right_to_left         |         61   |             47.8 |            13.2 |
|  6 |    3889637 | Jonathan dos Santos Ramírez     | right_to_left         |         47.6 |             46.5 |             1.1 |
|  8 |    3889637 | Richard Rafael Sánchez Guerrero | right_to_left         |         45.9 |             39.7 |             6.2 |
| 11 |    3889637 | Kevin Nahin Álvarez Campos      | right_to_left         |         39.7 |             27.6 |            12.1 |
| 14 |    3889637 | Sebastián Enzo Cáceres Ramos    | right_to_left         |         27.6 |             37.8 |           -10.2 |
| 17 |    3889637 | Jonathan dos Santos Ramírez     | right_to_left         |         35.7

In [50]:
print(
    passes_progression["progression_x"].describe(
        percentiles=[.10, .25, .50, .75, .90]
    )
)

count    69255.000000
mean         0.055819
std         17.923124
min       -105.700000
10%        -18.200000
25%         -9.200000
50%          0.000000
75%          9.300000
90%         18.400000
max        106.300000
Name: progression_x, dtype: float64


In [51]:
print(
    "Pases hacia adelante:",
    (passes_progression["progression_x"] > 0).mean()
)

Pases hacia adelante: 0.49561764493538374


In [52]:
print(
    "Pases hacia atrás:",
    (passes_progression["progression_x"] < 0).mean()
)

Pases hacia atrás: 0.4951700238249946


Encontramos una distribución casi simétrica de los resultados.\
Por lo cual, nuestro siguiente paso es filtrar pases que no sean de tipo progresivo.

In [53]:
## Identificamos el largo de la cancha
bins = [-120, -30, -15, -5, 5, 15, 30, 120]

In [54]:
labels = [
    "Muy hacia atrás",
    "Hacia atrás",
    "Casi neutro negativo",
    "Neutro",
    "Casi neutro positivo",
    "Hacia adelante",
    "Muy hacia adelante"
]

In [55]:
passes_progression["progression_zone"] = pd.cut(
    passes_progression["progression_x"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [56]:
progression_distribution = (
    passes_progression["progression_zone"]
    .value_counts(sort=False)
    .to_frame("passes")
)
print(progression_distribution.sort_values(by='passes', ascending=False).to_markdown())

| progression_zone     |   passes |
|:---------------------|---------:|
| Neutro               |    18939 |
| Casi neutro positivo |    15627 |
| Casi neutro negativo |    15588 |
| Hacia adelante       |     7008 |
| Hacia atrás          |     6874 |
| Muy hacia atrás      |     2626 |
| Muy hacia adelante   |     2593 |


In [57]:
progression_distribution["percentage"] = (
    progression_distribution["passes"]
    / len(passes_progression)
    * 100
)
print(progression_distribution.round(2).sort_values(by='passes', ascending=False).to_markdown())

| progression_zone     |   passes |   percentage |
|:---------------------|---------:|-------------:|
| Neutro               |    18939 |        27.35 |
| Casi neutro positivo |    15627 |        22.56 |
| Casi neutro negativo |    15588 |        22.51 |
| Hacia adelante       |     7008 |        10.12 |
| Hacia atrás          |     6874 |         9.93 |
| Muy hacia atrás      |     2626 |         3.79 |
| Muy hacia adelante   |     2593 |         3.74 |


Definimos nuevamente el rango de pases en función del campo de juego del equipo

In [58]:
bins_2 = [0, 30, 60, 90, 120]

In [59]:
labels_2 = [
    "Defensiva",
    "Construcción",
    "Progresión",
    "Último tercio"
]

In [60]:
passes_progression["origin_zone"] = pd.cut(
    passes_progression["location_x"],
    bins=bins_2,
    labels=labels_2,
    include_lowest=True
)

In [61]:
zone_progression = (
    passes_progression
    .groupby("origin_zone", observed=True)
    .agg(
        passes=("progression_x", "count"),
        mean_progression=("progression_x", "mean"),
        median_progression=("progression_x", "median")
    )
    .reset_index()
)

In [62]:
print(zone_progression.round(2).sort_values(by='passes', ascending=False).to_markdown())

|    | origin_zone   |   passes |   mean_progression |   median_progression |
|---:|:--------------|---------:|-------------------:|---------------------:|
|  1 | Construcción  |    23613 |              -0.21 |                -0.2  |
|  2 | Progresión    |    23176 |               0.26 |                 0.17 |
|  0 | Defensiva     |    12161 |               0.3  |                 0.3  |
|  3 | Último tercio |    10305 |              -0.08 |                 0    |


In [63]:
passes_progression["significant_progression"] = (
    passes_progression["progression_x"] >= 15
)

passes_progression["significant_progression"]

,significant_progression
4,False
6,False
8,False
11,False
14,False
...,...
501316,False
501320,False
501323,False
501326,True


In [64]:
significant_by_zone = (
    passes_progression
    .groupby("origin_zone", observed=True)
    .agg(
        total_passes=("progression_x", "count"),
        progressive_passes=("significant_progression", "sum")
    )
    .reset_index()
)

In [65]:
significant_by_zone["progressive_pct"] = (
    significant_by_zone["progressive_passes"]
    / significant_by_zone["total_passes"]
    * 100
)

In [66]:
print(significant_by_zone.round(2).to_markdown())

|    | origin_zone   |   total_passes |   progressive_passes |   progressive_pct |
|---:|:--------------|---------------:|---------------------:|------------------:|
|  0 | Defensiva     |          12161 |                 2685 |             22.08 |
|  1 | Construcción  |          23613 |                 3400 |             14.4  |
|  2 | Progresión    |          23176 |                 2859 |             12.34 |
|  3 | Último tercio |          10305 |                  749 |              7.27 |


 Ahora identificamos los pases de acuerdo a la zona de destino

In [67]:
passes_progression["destination_zone"] = pd.cut(
    passes_progression["end_location_x"],
    bins=bins_2,
    labels=labels_2,
    include_lowest=True
)

In [68]:
progression_zones = (
    passes_progression[
        passes_progression["progression_x"] >= 15
    ]
    .groupby(
        ["origin_zone", "destination_zone"],
        observed=True
    )
    .size()
    .reset_index(name="progressive_passes")
)

print(progression_zones.to_markdown())

|    | origin_zone   | destination_zone   |   progressive_passes |
|---:|:--------------|:-------------------|---------------------:|
|  0 | Defensiva     | Defensiva          |                  475 |
|  1 | Defensiva     | Construcción       |                 1457 |
|  2 | Defensiva     | Progresión         |                  635 |
|  3 | Defensiva     | Último tercio      |                  118 |
|  4 | Construcción  | Defensiva          |                  699 |
|  5 | Construcción  | Construcción       |                  760 |
|  6 | Construcción  | Progresión         |                 1602 |
|  7 | Construcción  | Último tercio      |                  339 |
|  8 | Progresión    | Defensiva          |                   47 |
|  9 | Progresión    | Construcción       |                  797 |
| 10 | Progresión    | Progresión         |                  766 |
| 11 | Progresión    | Último tercio      |                 1249 |
| 12 | Último tercio | Construcción       |                   

Sin embargo, es importante definir el destino de los pases progresivos

In [69]:
progression_zones["pct_from_origin"] = (
    progression_zones
    .groupby("origin_zone")["progressive_passes"]
    .transform(lambda x: x / x.sum() * 100)
)

print(
    progression_zones
    .sort_values(
        ["origin_zone", "progressive_passes"],
        ascending=[True, False]
    )
    .round(2).to_markdown()
)

|    | origin_zone   | destination_zone   |   progressive_passes |   pct_from_origin |
|---:|:--------------|:-------------------|---------------------:|------------------:|
|  1 | Defensiva     | Construcción       |                 1457 |             54.26 |
|  2 | Defensiva     | Progresión         |                  635 |             23.65 |
|  0 | Defensiva     | Defensiva          |                  475 |             17.69 |
|  3 | Defensiva     | Último tercio      |                  118 |              4.39 |
|  6 | Construcción  | Progresión         |                 1602 |             47.12 |
|  5 | Construcción  | Construcción       |                  760 |             22.35 |
|  4 | Construcción  | Defensiva          |                  699 |             20.56 |
|  7 | Construcción  | Último tercio      |                  339 |              9.97 |
| 11 | Progresión    | Último tercio      |                 1249 |             43.69 |
|  9 | Progresión    | Construcción       |

/tmp/ipykernel_28079/3735276426.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("origin_zone")["progressive_passes"]


In [70]:
check = passes_progression[
    (passes_progression["progression_x"] >= 15) &
    (passes_progression["origin_zone"] == "Construcción") &
    (passes_progression["destination_zone"] == "Defensiva")
]

print(
    check[
        [
            "match_id",
            "period",
            "attacking_direction",
            "location_x",
            "end_location_x",
            "progression_x",
            "origin_zone",
            "destination_zone"
        ]
    ].head(15).to_markdown()
)

print("Total de casos:", len(check))

|      |   match_id |   period | attacking_direction   |   location_x |   end_location_x |   progression_x | origin_zone   | destination_zone   |
|-----:|-----------:|---------:|:----------------------|-------------:|-----------------:|----------------:|:--------------|:-------------------|
|   24 |    3889637 |        1 | right_to_left         |         32.2 |             17   |            15.2 | Construcción  | Defensiva          |
|   93 |    3889637 |        1 | right_to_left         |         38   |             22   |            16   | Construcción  | Defensiva          |
|  129 |    3889637 |        1 | right_to_left         |         36.7 |             11.6 |            25.1 | Construcción  | Defensiva          |
|  674 |    3889637 |        1 | right_to_left         |         52.7 |             26.3 |            26.4 | Construcción  | Defensiva          |
|  723 |    3889637 |        1 | right_to_left         |         56.1 |             28.2 |            27.9 | Construcción  |

Normalizamos coordenadas longitudinales para establecer de manera correcta el avance del equipo en función de su posición en el campo.

In [71]:
passes_progression["start_x_norm"] = np.where(
    passes_progression["attacking_direction"] == "left_to_right",
    passes_progression["location_x"],
    120 - passes_progression["location_x"]
)

In [72]:
passes_progression["end_x_norm"] = np.where(
    passes_progression["attacking_direction"] == "left_to_right",
    passes_progression["end_location_x"],
    120 - passes_progression["end_location_x"]
)

In [73]:
passes_progression["progression_x_check"] = (
    passes_progression["end_x_norm"]
    - passes_progression["start_x_norm"]
)

In [74]:
print(
    passes_progression[
        ["progression_x", "progression_x_check"]
    ].describe().round(2).to_markdown()
)

|       |   progression_x |   progression_x_check |
|:------|----------------:|----------------------:|
| count |        69255    |              69255    |
| mean  |            0.06 |                  0.06 |
| std   |           17.92 |                 17.92 |
| min   |         -105.7  |               -105.7  |
| 25%   |           -9.2  |                 -9.2  |
| 50%   |            0    |                  0    |
| 75%   |            9.3  |                  9.3  |
| max   |          106.3  |                106.3  |


In [75]:
print(
    "Diferencias entre cálculos:",
    (
        passes_progression["progression_x"]
        - passes_progression["progression_x_check"]
    ).abs().gt(0.01).sum()
)

Diferencias entre cálculos: 0


Recalculamos en base a datos normalizaodos

In [76]:
bins_x = [0, 30, 60, 90, 120]

labels_x = [
    "Defensiva",
    "Construcción",
    "Progresión",
    "Último tercio"
]

In [77]:
passes_progression["origin_zone"] = pd.cut(
    passes_progression["start_x_norm"],
    bins=bins_x,
    labels=labels_x,
    include_lowest=True
)

passes_progression["destination_zone"] = pd.cut(
    passes_progression["end_x_norm"],
    bins=bins_x,
    labels=labels_x,
    include_lowest=True
)

In [78]:
# para checar posibles contradicciones que pudiera tener la normalización
check = passes_progression[
    (passes_progression["progression_x"] >= 15) &
    (
        passes_progression["destination_zone"].map(
            {label: i for i, label in enumerate(labels_x)}
        )
        <
        passes_progression["origin_zone"].map(
            {label: i for i, label in enumerate(labels_x)}
        )
    )
]

In [79]:
print("Pases >=15 m que retroceden de zona:", len(check))

Pases >=15 m que retroceden de zona: 0


In [80]:
progression_zones = (
    passes_progression[
        passes_progression["progression_x"] >= 15
    ]
    .groupby(
        ["origin_zone", "destination_zone"],
        observed=True
    )
    .size()
    .reset_index(name="progressive_passes")
)

In [81]:
progression_zones["pct_from_origin"] = (
    progression_zones["progressive_passes"]
    / progression_zones.groupby("origin_zone")[
        "progressive_passes"
    ].transform("sum")
    * 100
)

/tmp/ipykernel_28079/1968090656.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  / progression_zones.groupby("origin_zone")[


In [82]:
print(
    progression_zones
    .sort_values(
        ["origin_zone", "progressive_passes"],
        ascending=[True, False]
    )
    .round(2)
    .to_markdown()
)

|    | origin_zone   | destination_zone   |   progressive_passes |   pct_from_origin |
|---:|:--------------|:-------------------|---------------------:|------------------:|
|  1 | Defensiva     | Construcción       |                 1724 |             60.96 |
|  2 | Defensiva     | Progresión         |                  642 |             22.7  |
|  0 | Defensiva     | Defensiva          |                  344 |             12.16 |
|  3 | Defensiva     | Último tercio      |                  118 |              4.17 |
|  5 | Construcción  | Progresión         |                 2402 |             68.69 |
|  4 | Construcción  | Construcción       |                  709 |             20.27 |
|  6 | Construcción  | Último tercio      |                  386 |             11.04 |
|  8 | Progresión    | Último tercio      |                 1952 |             70.6  |
|  7 | Progresión    | Progresión         |                  813 |             29.4  |
|  9 | Último tercio | Último tercio      |

En base a los resultados podemos señalar:
- La mayoría de los pases mayores a 15 m de la zona defensiva se generan hacia la zona de construcción (60.96%)
- Esa misma tendencia se observa desde la zona de construcción a la de progresión (68.96%)
- La conexión más frecuente de los pases desde la zona de progresión es hacia el último tercio (70.60%)
- Los resultados nos muestran una construcción de juego escalonado
 $$ \text{Zona defensiva} → \text{Construcción} → \text{Progresión} → \text{Último tercio}$$

 Intentaremos entonces validar la recepción del pase por parte del equipo

In [83]:
cols_check = [
    c for c in events.columns
    if any(term in c.lower() for term in [
        "pass_outcome", "receipt", "outcome",
        "recipient", "possession", "index",
        "event_type", "team_name", "player_name"
    ])
]

In [84]:
print("Columnas relevantes:")
print(cols_check)

Columnas relevantes:
['index', 'event_type_id', 'event_type_name', 'possession', 'possession_team_id', 'possession_team_name', 'team_name', 'player_name', 'substituted_player_name', 'pass_recipient_id', 'pass_recipient_name', 'outcome_id', 'outcome_name', 'freeze_frame_player_name', 'formation_player_name', 'is_controlled_possession', 'player_possession_directness', 'possession_directness']


In [85]:
print("\nTipos de evento:")
print(
    events.loc[
        events["possession_team_id"] == TEAM_ID,
        "event_type_name"
    ].value_counts().head(20).to_markdown()
)


Tipos de evento:
| event_type_name   |   count |
|:------------------|--------:|
| Pass              |   69255 |
| Ball Receipt*     |   64377 |
| Carries           |   57144 |
| Shot              |   26258 |
| Pressure          |   22089 |
| Ball Recovery     |    5865 |
| Duel              |    3800 |
| Tactical Shift    |    3077 |
| Clearance         |    2404 |
| Block             |    2265 |
| Goal Keeper       |    2106 |
| Dribble           |    1886 |
| Foul Committed    |    1828 |
| Miscontrol        |    1776 |
| Foul Won          |    1746 |
| Dispossessed      |    1627 |
| Dribbled Past     |    1100 |
| Interception      |     970 |
| Substitution      |     653 |
| Injury Stoppage   |     475 |


In [86]:
cols_sample = [
    c for c in [
        "match_id", "index", "possession", "event_type_name",
        "team_name", "player_name", "pass_recipient_name",
        "pass_outcome_name", "location_x", "end_location_x"
    ]
    if c in events.columns
]

In [87]:
print("\nEjemplo de pases con avance >= 15:")
print(
    passes_progression.loc[
        passes_progression["progression_x"] >= 15,
        [c for c in cols_sample if c in passes_progression.columns]
    ].head(10).to_markdown()
)


Ejemplo de pases con avance >= 15:
|     |   match_id |   index |   possession | event_type_name   | team_name   | player_name                    | pass_recipient_name             |   location_x |   end_location_x |
|----:|-----------:|--------:|-------------:|:------------------|:------------|:-------------------------------|:--------------------------------|-------------:|-----------------:|
|  24 |    3889637 |      25 |            2 | Pass              | América     | Jonathan dos Santos Ramírez    | Sebastián Enzo Cáceres Ramos    |         32.2 |             17   |
|  58 |    3889637 |      59 |            2 | Pass              | América     | Román Martínez                 | Jonathan dos Santos Ramírez     |         97   |             81.4 |
|  93 |    3889637 |      94 |            4 | Pass              | América     | Jonathan dos Santos Ramírez    | Sebastián Enzo Cáceres Ramos    |         38   |             22   |
| 129 |    3889637 |     130 |            5 | Pass         

In [88]:
receipts = events[
    events["event_type_name"].astype(str).str.contains(
        "Ball Receipt", case=False, na=False
    )
]

print("\nEventos de recepción:", len(receipts))
print(receipts[cols_sample].head(10).to_markdown())


Eventos de recepción: 116392
|    |   match_id |   index |   possession | event_type_name   | team_name   | player_name                     |   pass_recipient_name |   location_x |   end_location_x |
|---:|-----------:|--------:|-------------:|:------------------|:------------|:--------------------------------|----------------------:|-------------:|-----------------:|
|  5 |    3889637 |       6 |            2 | Ball Receipt*     | América     | Jonathan dos Santos Ramírez     |                   nan |         47.8 |              nan |
|  7 |    3889637 |       8 |            2 | Ball Receipt*     | América     | Richard Rafael Sánchez Guerrero |                   nan |         46.5 |              nan |
|  9 |    3889637 |      10 |            2 | Ball Receipt*     | América     | Kevin Nahin Álvarez Campos      |                   nan |         39.7 |              nan |
| 12 |    3889637 |      13 |            2 | Ball Receipt*     | América     | Sebastián Enzo Cáceres Ramos    |   

In [89]:
# Orden cronológico de los eventos
ev = events.sort_values(["match_id", "index"]).copy()

In [90]:
# Inspeccionar outcome_name en las recepciones
receipts = ev[ev["event_type_name"] == "Ball Receipt*"]


print("Resultados registrados en Ball Receipt*:")
if "outcome_name" in receipts.columns:
    print(receipts["outcome_name"].value_counts(dropna=False).head(10).to_markdown())

Resultados registrados en Ball Receipt*:
| outcome_name   |   count |
|:---------------|--------:|
| nan            |  105266 |
| Incomplete     |   11126 |


In [91]:
group_cols = ["match_id", "possession"]

for col in ["event_type_name", "team_name", "player_name"]:
    ev[f"next_{col}"] = ev.groupby(group_cols)[col].shift(-1)

ev.head()

,id,index,match_id,period,timestamp,minute,second,event_type_id,event_type_name,possession,...,america_score,opponent_score,coach,stadium,match_status,match_status_360,result,next_event_type_name,next_team_name,next_player_name
0,ab861d54-b8b8-575b-ad0a-305b4798cede,1,3889637,1,00:00,0,0,35,Starting XI,1,...,1,2,André Soares Jardine,Estadio Azteca,available,available,L,Starting XI,América,NaN
4163,ab861d54-b8b8-575b-ad0a-305b4798cede,1,3889637,1,00:00,0,0,35,Starting XI,1,...,1,2,André Soares Jardine,Estadio Azteca,available,available,L,Starting XI,América,NaN
4164,ab861d54-b8b8-575b-ad0a-305b4798cede,1,3889637,1,00:00,0,0,35,Starting XI,1,...,1,2,André Soares Jardine,Estadio Azteca,available,available,L,Starting XI,América,NaN
4165,ab861d54-b8b8-575b-ad0a-305b4798cede,1,3889637,1,00:00,0,0,35,Starting XI,1,...,1,2,André Soares Jardine,Estadio Azteca,available,available,L,Starting XI,América,NaN
4166,ab861d54-b8b8-575b-ad0a-305b4798cede,1,3889637,1,00:00,0,0,35,Starting XI,1,...,1,2,André Soares Jardine,Estadio Azteca,available,available,L,Starting XI,América,NaN


In [92]:
# Comparar pase y evento inmediatamente posterior
pass_events = ev[ev["event_type_name"] == "Pass"].copy()

In [93]:
pass_events["next_is_receipt"] = (
    pass_events["next_event_type_name"] == "Ball Receipt*"
)

In [94]:
pass_events["recipient_matches"] = (
    pass_events["pass_recipient_name"].notna()
    & pass_events["next_is_receipt"]
    & (
        pass_events["pass_recipient_name"]
        == pass_events["next_player_name"]
    )
)

# pass_events["recipient_matches"]

In [95]:
print("\nValidación de todos los pases del equipo:")
print("Total de pases:", len(pass_events))
print("Seguidos inmediatamente por Ball Receipt*:",
      pass_events["next_is_receipt"].sum())
print("Recepción con destinatario coincidente:",
      pass_events["recipient_matches"].sum())


Validación de todos los pases del equipo:
Total de pases: 126618
Seguidos inmediatamente por Ball Receipt*: 103604
Recepción con destinatario coincidente: 103604


In [96]:
# Aplicar el análisis a los pases con avance >= 15
progressive_ids = passes_progression.loc[
    passes_progression["progression_x"] >= 15,
    ["match_id", "index"]
].drop_duplicates()

In [97]:
progressive_check = pass_events.merge(
    progressive_ids.assign(selected_progression=True),
    on=["match_id", "index"],
    how="inner",
    validate="one_to_one"
)

In [98]:
print("\nPases con avance >= 15:")
print("Total:", len(progressive_check))
print("Seguidos por Ball Receipt*:",
      progressive_check["next_is_receipt"].sum())
print("Recepción con destinatario coincidente:",
      progressive_check["recipient_matches"].sum())


Pases con avance >= 15:
Total: 9693
Seguidos por Ball Receipt*: 7353
Recepción con destinatario coincidente: 7353


In [99]:
print(
    progressive_check[
        [
            "match_id", "index", "possession",
            "player_name", "pass_recipient_name",
            "next_event_type_name", "next_player_name",
            "next_is_receipt", "recipient_matches"
        ]
    ].head(15).to_markdown()
)

|    |   match_id |   index |   possession | player_name                    | pass_recipient_name             | next_event_type_name   | next_player_name                | next_is_receipt   | recipient_matches   |
|---:|-----------:|--------:|-------------:|:-------------------------------|:--------------------------------|:-----------------------|:--------------------------------|:------------------|:--------------------|
|  0 |    3889637 |      25 |            2 | Jonathan dos Santos Ramírez    | Sebastián Enzo Cáceres Ramos    | Ball Receipt*          | Sebastián Enzo Cáceres Ramos    | True              | True                |
|  1 |    3889637 |      59 |            2 | Román Martínez                 | Jonathan dos Santos Ramírez     | Pressure               | Diego Chávez Collins            | False             | False               |
|  2 |    3889637 |      94 |            4 | Jonathan dos Santos Ramírez    | Sebastián Enzo Cáceres Ramos    | Ball Receipt*          | Sebastián E

In [100]:
progressive_check.next_event_type_name.unique()

array(['Ball Receipt*', 'Pressure', nan, 'Substitution', 'Tactical Shift',
       'Duel', 'Clearance', 'Player On', 'Pass', 'Foul Committed',
       'Goal Keeper', 'Ball Recovery', 'Interception', 'Shield', 'Block',
       'Injury Stoppage', 'Referee Ball-Drop', 'Player Off'], dtype=object)

In [101]:
# Ordenar eventos y trabajar con posesiones de América
ev = events.sort_values(["match_id", "index"]).copy()


In [102]:
ev_america_possessions = ev[
    ev["possession_team_id"] == TEAM_ID
].copy()

ev_america_possessions.head()

,id,index,match_id,period,timestamp,minute,second,event_type_id,event_type_name,possession,...,match_week,opponent,venue,america_score,opponent_score,coach,stadium,match_status,match_status_360,result
4,7d53e561-53d8-51db-9e4e-3ca9427cc059,5,3889637,1,00:00:00.044,0,0,30,Pass,2,...,1,Juárez,Home,1,2,André Soares Jardine,Estadio Azteca,available,available,L
5,5630a90f-4dff-5016-951d-1505c2cb1ca5,6,3889637,1,00:00:02.268,0,2,42,Ball Receipt*,2,...,1,Juárez,Home,1,2,André Soares Jardine,Estadio Azteca,available,available,L
6,1344c9dc-12fd-52ed-b80f-ec70d507b891,7,3889637,1,00:00:02.598,0,2,30,Pass,2,...,1,Juárez,Home,1,2,André Soares Jardine,Estadio Azteca,available,available,L
7,fd85acee-75f9-5918-8db0-2c246f5589a7,8,3889637,1,00:00:03.440,0,3,42,Ball Receipt*,2,...,1,Juárez,Home,1,2,André Soares Jardine,Estadio Azteca,available,available,L
8,17d605bc-ed2c-57e4-919d-4a16dc84eec0,9,3889637,1,00:00:04.310,0,4,30,Pass,2,...,1,Juárez,Home,1,2,André Soares Jardine,Estadio Azteca,available,available,L


In [103]:
# Pases seleccionados: avance longitudinal >= 15
selected = passes_progression.loc[
    passes_progression["progression_x"] >= 15,
    ["match_id", "index", "possession", "pass_recipient_name"]
].drop_duplicates(["match_id", "index"]).copy()

In [104]:
selected

,match_id,index,possession,pass_recipient_name
24,3889637,25,2,Sebastián Enzo Cáceres Ramos
58,3889637,59,2,Jonathan dos Santos Ramírez
93,3889637,94,4,Sebastián Enzo Cáceres Ramos
129,3889637,130,5,Óscar Francisco Jiménez Fabela
168,3889637,169,5,Richard Rafael Sánchez Guerrero
...,...,...,...,...
501279,4057783,2750,167,Henry Josué Martín Mex
501292,4057783,2763,169,Paul Brian Rodríguez Bravo
501314,4057783,2785,171,NaN
501326,4057783,2797,174,Miguel Ángel Vázquez García


In [105]:
# Revisar cada pase hasta el siguiente pase de la misma posesión
event_groups = {
    key: group.reset_index(drop=True)
    for key, group in ev_america_possessions.groupby(
        ["match_id", "possession"], sort=False
    )
}

In [106]:
results = []

for row in selected.itertuples(index=False):
    key = (row.match_id, row.possession)
    group = event_groups.get(key)

    if group is None:
        results.append((False, False, 0))
        continue

    positions = group.index[group["index"] == row.index].tolist()

    if not positions:
        results.append((False, False, 0))
        continue

    pos = positions[0]
    following = group.iloc[pos + 1:]

    # Detenerse antes del siguiente pase
    next_pass_positions = following.index[
        following["event_type_name"] == "Pass"
    ].tolist()

    if next_pass_positions:
        following = following.loc[:next_pass_positions[0] - 1]

    receipt_events = following[
        following["event_type_name"] == "Ball Receipt*"
    ]

    recipient_found = (
        receipt_events["player_name"].eq(
            row.pass_recipient_name
        ).any()
        if pd.notna(row.pass_recipient_name)
        else False
    )

    results.append((
        len(receipt_events) > 0,
        recipient_found,
        len(following)
    ))


In [107]:
len(results)

9693

In [108]:
selected["receipt_before_next_pass"] = [r[0] for r in results]
selected["recipient_found"] = [r[1] for r in results]
selected["intermediate_events"] = [r[2] for r in results]

In [109]:
print("Pases seleccionados:", len(selected))
print(
    "Con recepción antes del siguiente pase:",
    selected["receipt_before_next_pass"].sum()
)
print(
    "Con recepción del destinatario esperado:",
    selected["recipient_found"].sum()
)

print("\nPorcentajes:")
print(
    (selected[
        ["receipt_before_next_pass", "recipient_found"]
    ].mean() * 100).round(2).rename("Porcentaje (%)").to_markdown()
)


Pases seleccionados: 9693
Con recepción antes del siguiente pase: 8604
Con recepción del destinatario esperado: 8604

Porcentajes:
|                          |   Porcentaje (%) |
|:-------------------------|-----------------:|
| receipt_before_next_pass |            88.77 |
| recipient_found          |            88.77 |


In [110]:
print("\nCasos con eventos intermedios:")
print(
    selected.loc[
        selected["intermediate_events"] > 0,
        [
            "match_id", "index", "possession",
            "pass_recipient_name", "receipt_before_next_pass",
            "recipient_found", "intermediate_events"
        ]
    ].head(15).to_markdown()
)


Casos con eventos intermedios:
|     |   match_id |   index |   possession | pass_recipient_name             | receipt_before_next_pass   | recipient_found   |   intermediate_events |
|----:|-----------:|--------:|-------------:|:--------------------------------|:---------------------------|:------------------|----------------------:|
|  24 |    3889637 |      25 |            2 | Sebastián Enzo Cáceres Ramos    | True                       | True              |                     2 |
|  58 |    3889637 |      59 |            2 | Jonathan dos Santos Ramírez     | True                       | True              |                     2 |
|  93 |    3889637 |      94 |            4 | Sebastián Enzo Cáceres Ramos    | True                       | True              |                     2 |
| 129 |    3889637 |     130 |            5 | Óscar Francisco Jiménez Fabela  | True                       | True              |                     1 |
| 168 |    3889637 |     169 |            5 | Rich

In [111]:
ev = events.sort_values(["match_id", "index"]).copy()
group_cols = ["match_id", "possession"]

# Siguiente evento de la misma posesión
ev["next_event_type"] = ev.groupby(group_cols)["event_type_name"].shift(-1)
ev["next_team"] = ev.groupby(group_cols)["team_name"].shift(-1)
ev["next_controlled"] = ev.groupby(group_cols)[
    "is_controlled_possession"
].shift(-1)

# Recepciones registradas en posesiones de, equipo
receipts = ev[
    (ev["event_type_name"] == "Ball Receipt*")
    & (ev["possession_team_name"] == "América")
].copy()



In [112]:
print("Total de recepciones:", len(receipts))

print("\nControl registrado en la recepción:")
print(
    receipts["is_controlled_possession"]
    .value_counts(dropna=False)
    .to_frame("eventos").to_markdown()
)



Total de recepciones: 64377

Control registrado en la recepción:
| is_controlled_possession   |   eventos |
|:---------------------------|----------:|
| True                       |     47133 |
| nan                        |     10285 |
| False                      |      4907 |
| true                       |       616 |
| midfield_third             |       572 |
| final_third                |       511 |
| defensive_third            |       229 |
| false                      |       124 |


In [113]:
receipts.is_controlled_possession.unique()

array([True, False, nan, 'true', 'false', 'midfield_third',
       'defensive_third', 'final_third'], dtype=object)

In [114]:
print("\nTipo de evento posterior:")
print(
    receipts["next_event_type"]
    .value_counts(dropna=False)
    .head(15)
    .to_frame("eventos").to_markdown()
)

print("\nEquipo que realiza el evento posterior:")
print(
    receipts["next_team"]
    .value_counts(dropna=False)
    .head(10)
    .to_frame("eventos").to_markdown()
)

print("\nControl registrado en el evento posterior:")
print(
    receipts["next_controlled"]
    .value_counts(dropna=False)
    .to_frame("eventos").to_markdown()
)



Tipo de evento posterior:
| next_event_type   |   eventos |
|:------------------|----------:|
| Carries           |     49856 |
| Pass              |      7561 |
| nan               |      3049 |
| Clearance         |       739 |
| Shot              |       548 |
| Block             |       520 |
| Interception      |       427 |
| Pressure          |       385 |
| Miscontrol        |       358 |
| Duel              |       314 |
| Ball Recovery     |       188 |
| Foul Committed    |       162 |
| Injury Stoppage   |        80 |
| Goal Keeper       |        43 |
| Substitution      |        39 |

Equipo que realiza el evento posterior:
| next_team     |   eventos |
|:--------------|----------:|
| América       |     58344 |
| nan           |      3049 |
| Cruz Azul     |       289 |
| Pachuca       |       271 |
| León          |       198 |
| Guadalajara   |       197 |
| Tigres UANL   |       192 |
| Toluca        |       191 |
| Atlas         |       177 |
| Santos Laguna |       

In [115]:
# Diagnóstico: confirmar qué representa realmente la columna
print("Tipo de dato:")
print(events["is_controlled_possession"].dtype)

print("\nValores más frecuentes:")
print(
    events["is_controlled_possession"]
    .value_counts(dropna=False)
    .head(15)
    .to_frame("conteo").to_markdown()
)

Tipo de dato:
object

Valores más frecuentes:
| is_controlled_possession   |   conteo |
|:---------------------------|---------:|
| True                       |   295052 |
| nan                        |   134633 |
| False                      |    54053 |
| true                       |     4544 |
| midfield_third             |     4067 |
| final_third                |     3872 |
| defensive_third            |     2774 |
| false                      |     1036 |
| 0.0017974227               |       21 |
| 0.0091792364               |       20 |
| 0.0133826835               |       19 |
| 0.0589727312               |       19 |
| 0.005055401                |       18 |
| 0.0017451613               |       18 |
| 0.0103466902               |       17 |


In [116]:
ev = events.sort_values(["match_id", "index"]).copy()

groups = {
    key: group.reset_index(drop=True)
    for key, group in ev.groupby(
        ["match_id", "possession"], sort=False
    )
}

continuity_results = []

for row in selected.itertuples(index=False):
    key = (row.match_id, row.possession)
    group = groups.get(key)

    if group is None:
        continuity_results.append((False, None, None))
        continue

    positions = group.index[group["index"] == row.index].tolist()

    if not positions:
        continuity_results.append((False, None, None))
        continue

    pos = positions[0]
    following = group.iloc[pos + 1:]

    # Buscar recepción del destinatario antes del siguiente pase
    next_pass = following.index[
        following["event_type_name"] == "Pass"
    ].tolist()

    if next_pass:
        following = following.loc[:next_pass[0] - 1]

    receipts = following[
        (following["event_type_name"] == "Ball Receipt*")
        & (following["player_name"] == row.pass_recipient_name)
    ]

    if receipts.empty:
        continuity_results.append((False, None, None))
        continue

    receipt_idx = receipts.index[0]
    receipt_pos = group.index.get_loc(receipt_idx)

    if receipt_pos + 1 >= len(group):
        continuity_results.append((True, None, None))
        continue

    next_action = group.iloc[receipt_pos + 1]

    continuity_results.append((
        True,
        next_action["team_name"],
        next_action["event_type_name"]
    ))


In [117]:
selected["receipt_found"] = [r[0] for r in continuity_results]
selected["team_after_receipt"] = [r[1] for r in continuity_results]
selected["action_after_receipt"] = [r[2] for r in continuity_results]

confirmed = selected[selected["receipt_found"]].copy()

print("\nPases con recepción del destinatario:", len(confirmed))

print("\nEquipo de la acción posterior a la recepción:")
print(
    confirmed["team_after_receipt"]
    .value_counts(dropna=False)
    .to_frame("conteo").to_markdown()
)

print("\nTipo de acción posterior:")
print(
    confirmed["action_after_receipt"]
    .value_counts(dropna=False)
    .head(12)
    .to_frame("conteo").to_markdown()
)

continuity_rate = confirmed["team_after_receipt"].eq("América").mean()

print(
    "\nProporción con siguiente evento atribuido a América:",
    f"{continuity_rate:.2%}"
)



Pases con recepción del destinatario: 8604

Equipo de la acción posterior a la recepción:
| team_after_receipt   |   conteo |
|:---------------------|---------:|
| América              |     7187 |
|                      |      817 |
| Cruz Azul            |       67 |
| Pachuca              |       54 |
| Guadalajara          |       46 |
| Toluca               |       43 |
| Monterrey            |       40 |
| Tigres UANL          |       39 |
| Tijuana              |       37 |
| León                 |       37 |
| Pumas UNAM           |       33 |
| Atlas                |       33 |
| Puebla               |       29 |
| Santos Laguna        |       28 |
| Mazatlán             |       26 |
| Atlético San Luis    |       25 |
| Necaxa               |       24 |
| Querétaro            |       23 |
| Juárez               |       16 |

Tipo de acción posterior:
| action_after_receipt   |   conteo |
|:-----------------------|---------:|
| Carries                |     5891 |
| Pass      